In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("/content/sample_superstore_clean.csv")

print("Dataset dimensions:", df.shape)
df.head()

Dataset dimensions: (9994, 22)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Discount Group
0,1,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,Below 30%
1,2,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,Below 30%
2,3,CA-2020-138688,2020-06-12,2020-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,Below 30%
3,4,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,30% or Higher
4,5,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,Below 30%


In [2]:
conn = sqlite3.connect("retail_sales.db")

print("Database connection created successfully.")

Database connection created successfully.


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("/", "_")
)

df.columns.tolist()

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country_region',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit',
 'discount_group']

In [4]:
df.to_sql(
    "retail_sales",
    conn,
    if_exists="replace",
    index=False
)

print("Retail sales table created successfully.")

Retail sales table created successfully.


In [5]:
query = """
SELECT COUNT(*) AS total_rows
FROM retail_sales;
"""

pd.read_sql_query(query, conn)

,total_rows
0,9994


In [6]:
query = """
SELECT
    category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(
        SUM(profit) / NULLIF(SUM(sales), 0) * 100,
        2
    ) AS profit_margin_pct
FROM retail_sales
GROUP BY category
ORDER BY total_profit DESC;
"""

category_sql = pd.read_sql_query(query, conn)

category_sql

,category,total_sales,total_profit,profit_margin_pct
0,Technology,836154.03,145454.95,17.40
1,Office Supplies,719047.03,122490.80,17.04
2,Furniture,741999.80,18451.27,2.49


## 1. Category Profitability Analysis

SQL aggregation confirms substantial profitability differences across product categories. Technology generates the greatest total profit and strongest overall margin, while Office Supplies also performs well. Furniture generates substantial revenue but produces a significantly weaker profit margin, indicating a need for deeper product-level investigation.

In [7]:
query = """
WITH subcategory_profit AS (
    SELECT
        sub_category,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit
    FROM retail_sales
    GROUP BY sub_category
)

SELECT
    sub_category,
    total_sales,
    total_profit,
    RANK() OVER (
        ORDER BY total_profit DESC
    ) AS profit_rank
FROM subcategory_profit
ORDER BY profit_rank;
"""

product_rank_sql = pd.read_sql_query(query, conn)

product_rank_sql

,sub_category,total_sales,total_profit,profit_rank
0,Copiers,149528.03,55617.82,1
1,Phones,330007.05,44515.73,2
2,Accessories,167380.32,41936.64,3
3,Paper,78479.21,34053.57,4
4,Binders,203412.73,30221.76,5
5,Chairs,328449.10,26590.17,6
6,Storage,223843.61,21278.83,7
7,Appliances,107532.16,18138.01,8
8,Furnishings,91705.16,13059.14,9
9,Envelopes,16476.40,6964.18,10


In [8]:
query = """
WITH high_discount_products AS (
    SELECT
        sub_category,
        COUNT(*) AS transactions,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(AVG(discount), 3) AS average_discount
    FROM retail_sales
    WHERE discount >= 0.30
    GROUP BY sub_category
)

SELECT
    sub_category,
    transactions,
    total_sales,
    total_profit,
    average_discount,
    RANK() OVER (
        ORDER BY total_profit
    ) AS loss_rank
FROM high_discount_products
ORDER BY loss_rank;
"""

high_discount_sql = pd.read_sql_query(query, conn)

high_discount_sql

,sub_category,transactions,total_sales,total_profit,average_discount,loss_rank
0,Binders,613,36140.61,-38510.50,0.738,1
1,Tables,176,89956.54,-30698.22,0.393,2
2,Machines,53,76839.11,-29555.35,0.543,3
3,Bookcases,70,28544.00,-11097.76,0.445,4
4,Appliances,67,3382.53,-8629.64,0.800,5
5,Chairs,158,70005.50,-6737.12,0.300,6
6,Phones,109,34337.35,-6385.79,0.400,7
7,Furnishings,138,6644.70,-5944.66,0.600,8
8,Copiers,9,16919.80,2182.98,0.400,9


In [9]:
query = """
WITH customer_summary AS (
    SELECT
        customer_id,
        customer_name,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        COUNT(DISTINCT order_id) AS total_orders
    FROM retail_sales
    GROUP BY customer_id, customer_name
)

SELECT
    customer_id,
    customer_name,
    total_sales,
    total_profit,
    total_orders,
    RANK() OVER (
        ORDER BY total_sales DESC
    ) AS sales_rank
FROM customer_summary
ORDER BY sales_rank
LIMIT 10;
"""

top_customers_sql = pd.read_sql_query(query, conn)

top_customers_sql

,customer_id,customer_name,total_sales,total_profit,total_orders,sales_rank
0,SM-20320,Sean Miller,25043.05,-1980.74,5,1
1,TC-20980,Tamara Chand,19052.22,8981.32,5,2
2,RB-19360,Raymond Buch,15117.34,6976.10,6,3
3,TA-21385,Tom Ashbrook,14595.62,4703.79,4,4
4,AB-10105,Adrian Barton,14473.57,5444.81,10,5
5,KL-16645,Ken Lonsdale,14175.23,806.85,12,6
6,SC-20095,Sanjit Chand,14142.33,5757.41,9,7
7,HL-15040,Hunter Lopez,12873.30,5622.43,6,8
8,SE-20110,Sanjit Engle,12209.44,2650.68,11,9
9,CC-12370,Christopher Conant,12129.07,2177.05,5,10


In [10]:
query = """
WITH customer_summary AS (
    SELECT
        customer_id,
        customer_name,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        COUNT(DISTINCT order_id) AS total_orders
    FROM retail_sales
    GROUP BY customer_id, customer_name
)

SELECT
    customer_id,
    customer_name,
    total_sales,
    total_profit,
    total_orders
FROM customer_summary
WHERE total_profit < 0
ORDER BY total_sales DESC
LIMIT 10;
"""

loss_customers_sql = pd.read_sql_query(query, conn)

loss_customers_sql

,customer_id,customer_name,total_sales,total_profit,total_orders
0,SM-20320,Sean Miller,25043.05,-1980.74,5
1,BM-11140,Becky Martin,11789.63,-1659.96,4
2,GT-14635,Grant Thornton,9351.21,-4108.66,3
3,PF-19120,Peter Fuller,9062.86,-614.29,4
4,NF-18385,Natalie Fritzler,8322.83,-1695.97,7
5,SB-20290,Sean Braxton,8057.89,-2082.75,7
6,ZC-21910,Zuschuss Carroll,8025.71,-1032.15,13
7,JH-15985,Joseph Holt,7955.00,-644.70,6
8,JA-15970,Joseph Airdo,6491.03,-819.42,8
9,VW-21775,Victoria Wilson,6134.04,-874.66,10


In [11]:
query = """
SELECT
    region,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(
        SUM(profit) / NULLIF(SUM(sales), 0) * 100,
        2
    ) AS profit_margin_pct
FROM retail_sales
GROUP BY region
ORDER BY total_profit DESC;
"""

regional_sql = pd.read_sql_query(query, conn)

regional_sql

,region,total_sales,total_profit,total_orders,profit_margin_pct
0,West,725457.82,108418.45,1611,14.94
1,East,678781.24,91522.78,1401,13.48
2,South,391721.91,46749.43,822,11.93
3,Central,501239.89,39706.36,1175,7.92


In [12]:
query = """
WITH state_performance AS (
    SELECT
        state,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        ROUND(AVG(discount), 3) AS average_discount
    FROM retail_sales
    GROUP BY state
)

SELECT
    state,
    total_sales,
    total_profit,
    average_discount,
    ROUND(
        total_profit / NULLIF(total_sales, 0) * 100,
        2
    ) AS profit_margin_pct
FROM state_performance
WHERE total_profit < 0
ORDER BY total_profit
LIMIT 10;
"""

loss_states_sql = pd.read_sql_query(query, conn)

loss_states_sql

,state,total_sales,total_profit,average_discount,profit_margin_pct
0,Texas,170188.05,-25729.36,0.370,-15.12
1,Ohio,78258.14,-16971.38,0.325,-21.69
2,Pennsylvania,116511.91,-15559.96,0.329,-13.35
3,Illinois,80166.10,-12607.89,0.390,-15.73
4,North Carolina,55603.16,-7490.91,0.284,-13.47
5,Colorado,32108.12,-6527.86,0.316,-20.33
6,Tennessee,30661.87,-5341.69,0.291,-17.42
7,Arizona,35282.00,-3427.92,0.304,-9.72
8,Florida,89473.71,-3399.30,0.299,-3.80
9,Oregon,17431.15,-1190.47,0.289,-6.83
